<a href="https://colab.research.google.com/github/BRKodali/WeatherPredictingML/blob/main/BhargavKodaliWeatherPredictingML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
#@title Library Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [27]:
#@title Converting the CSV to a DataFrame

# Convert to DataFrame
df = pd.read_csv('WeatherDataSet.csv')

# Display the DataFrame
display(df)

,STATION,NAME,DATE,PRCP,SNOW,SNWD,TAVG,TMAX,TMIN
0,USC00050848,"BOULDER, CO US",1948-01-01,0.0,0.0,NaN,NaN,6.7,-14.4
1,USC00050848,"BOULDER, CO US",1948-01-02,0.0,0.0,NaN,NaN,11.1,-3.3
2,USC00050848,"BOULDER, CO US",1948-01-03,0.0,0.0,NaN,NaN,11.7,1.1
3,USC00050848,"BOULDER, CO US",1948-01-04,0.0,0.0,NaN,NaN,13.9,-2.2
4,USC00050848,"BOULDER, CO US",1948-01-05,0.0,0.0,NaN,NaN,12.2,-5.6
...,...,...,...,...,...,...,...,...,...
85868,USW00014922,"MINNEAPOLIS ST. PAUL INTERNATIONAL AIRPORT, MN US",2026-10-25,0.5,0.0,0.0,NaN,12.8,8.9
85869,USW00014922,"MINNEAPOLIS ST. PAUL INTERNATIONAL AIRPORT, MN US",2026-10-26,9.7,0.0,0.0,NaN,9.4,5.6
85870,USW00014922,"MINNEAPOLIS ST. PAUL INTERNATIONAL AIRPORT, MN US",2026-10-27,3.6,0.0,0.0,NaN,13.3,6.7
85871,USW00014922,"MINNEAPOLIS ST. PAUL INTERNATIONAL AIRPORT, MN US",2026-10-28,1.0,0.0,0.0,NaN,22.2,10.6


In [28]:
#@title Adding a lagged column
# Shift the data so the model can look at the data from not just today, but also yesterday
df['TMAX_Yesterday'] = df.groupby('NAME')['TMAX'].shift(1)
df['TMIN_Yesterday'] = df.groupby('NAME')['TMIN'].shift(1)

In [29]:
#@title Renaming Location Names for Readability
df['NAME'] = (df['NAME'].replace('BOULDER, CO US', 'BOULDER')
                        .replace("DULUTH INTERNATIONAL AIRPORT, MN US", "DULUTH")
                        .replace("MINNEAPOLIS ST. PAUL INTERNATIONAL AIRPORT, MN US", "MSP_INTERNATIONAL"))

In [30]:
#@title One-Hot Encoding
df = pd.get_dummies(df, columns=['NAME'], drop_first=True)

In [31]:
#@title Removes TAVG column
df.drop(columns = 'TAVG', inplace = True)

In [32]:
#@title Removing STATION Column
df.drop(columns = "STATION", inplace = True)

In [33]:
#@title Forward and Backward filling
# First, fill everything you can by copying forward
df.ffill(inplace=True)

# Then, catch any blanks at the very beginning by copying backward
df.bfill(inplace=True)

In [34]:
#@title Splitting Dates into 3 different columns
# Convert the text strings into pandas datetime objects
df['DATE'] = pd.to_datetime(df['DATE'])

# Extract the time components into new numerical columns
df['Year'] = df['DATE'].dt.year
df['Month'] = df['DATE'].dt.month
df['Day'] = df['DATE'].dt.day
df['DayOfYear'] = df['DATE'].dt.dayofyear

# Drop the original text column so scikit-learn does not crash
df.drop('DATE', axis=1, inplace=True)

In [35]:
#@title Training the Model
# 1. Define your multiple targets (outputs)
# We are predicting both High and Low temperatures
y = df[['TMAX', 'TMIN']]

# 2. Define your features (X)
# Make sure you drop the targets, plus TAVG (Average Temp) so you don't leak the answer
X = df.drop(['TMAX', 'TMIN',], axis=1)

# 3. Split chronologically
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 4. Train a multi-output model
# HistGradientBoosting requires a wrapper to handle multiple outputs at once
base_model = HistGradientBoostingRegressor()
multi_model = MultiOutputRegressor(base_model)

# Train on the multi-column y_train
multi_model.fit(X_train, y_train)

MultiOutputRegressor(estimator=HistGradientBoostingRegressor())

In [38]:
#@title Testing The Model
# 1. Generate predictions from your blind test set
predictions = multi_model.predict(X_test)

# 2. Calculate Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, predictions, multioutput='raw_values')

# 3. Calculate Mean Squared Error (MSE), then take the square root for RMSE
mse = mean_squared_error(y_test, predictions, multioutput='raw_values')
rmse = np.sqrt(mse)

# 4. Calculate R-squared
r2 = r2_score(y_test, predictions, multioutput='raw_values')

# Print results cleanly for TMAX (index 0) and TMIN (index 1)
print(f"TMAX: MAE: {mae[0]:.2f}, RMSE: {rmse[0]:.2f}, R-Squared: {r2[0]:.2f}")
print(f"TMIN: MAE: {mae[1]:.2f}, RMSE: {rmse[1]:.2f}, R-Squared: {r2[1]:.2f}")

TMAX: MAE: 3.06, RMSE: 4.00, R-Squared: 0.91
TMIN: MAE: 2.62, RMSE: 3.39, R-Squared: 0.92


In [37]:
#@title Display Dataframe for Debugging
display(df)

,PRCP,SNOW,SNWD,TMAX,TMIN,TMAX_Yesterday,TMIN_Yesterday,NAME_DULUTH,NAME_MSP_INTERNATIONAL,Year,Month,Day,DayOfYear
0,0.0,0.0,51.0,6.7,-14.4,6.7,-14.4,False,False,1948,1,1,1
1,0.0,0.0,51.0,11.1,-3.3,6.7,-14.4,False,False,1948,1,2,2
2,0.0,0.0,51.0,11.7,1.1,11.1,-3.3,False,False,1948,1,3,3
3,0.0,0.0,51.0,13.9,-2.2,11.7,1.1,False,False,1948,1,4,4
4,0.0,0.0,51.0,12.2,-5.6,13.9,-2.2,False,False,1948,1,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
85868,0.5,0.0,0.0,12.8,8.9,23.9,10.6,False,True,2026,10,25,298
85869,9.7,0.0,0.0,9.4,5.6,12.8,8.9,False,True,2026,10,26,299
85870,3.6,0.0,0.0,13.3,6.7,9.4,5.6,False,True,2026,10,27,300
85871,1.0,0.0,0.0,22.2,10.6,13.3,6.7,False,True,2026,10,28,301
